In [1]:
# ensures that if we change something in .core, it sees it imediately without having to restart the kernel
# every time you run a cell, all imported modules are reloaded auomatically
%load_ext autoreload
%autoreload 2 

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap
import scipy
from kneed import KneeLocator
from statsmodels.tsa.stattools import adfuller

# Our package
import neuro_lib as nl

# SETTING SEED FOR REPRODUCIBILITY
np.random.seed(0)

relations_dict = {"linear": lambda s: s,
                "tanh": lambda s: np.tanh(5*s),
                "quadratic": lambda s: 2 * s**2 - 1}

/Users/angelabortolato/Desktop/ITI_GroupProject/neuro_lib/analytics.py:22: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  normal variables with variances $\sigma^2$ and $v^2$.


# Data with temporal structure
In the previous section, we evaluated different Mutual Information estimators (KDE, binning, Gaussian, and Gaussian Copula) on i.i.d. variables with known functional dependencies (linear, quadratic, and non-linear transformations). This allowed us to assess their ability to capture static dependencies under controlled conditions.

We now extend the analysis to the temporal domain, where dependencies are directional and evolve over time. In this context, Transfer Entropy (TE) provides a natural generalization of MI, enabling the detection of directed information flow between time series.

To systematically evaluate the performance of TE estimators, we generate synthetic time series with known ground-truth causal structure. This allows us to benchmark the estimators under controlled settings before applying them to real neural data.


## Simulated time series

Two classes of systems are considered, representing complementary regimes:

1) Linear System: Autoregressive (AR) Processes
We generate coupled autoregressive processes to model linear interactions with a known directional structure. A typical formulation is:

\begin{aligned}
X_t &= \alpha X_{t-1} + \epsilon_t^X \\
Y_t &= \beta Y_{t-1} + \gamma X_{t-\ell} + \epsilon_t^Y
\end{aligned}

where:
*	$\alpha, \beta$ control the persistence of each process,
*	$\gamma$ determines the coupling strength from X to Y,
*	$\ell$ is the true interaction lag,
*	$\epsilon_t^X, \epsilon_t^Y$ are Gaussian noise terms.

This setup defines a clear ground-truth direction of information flow ($X \rightarrow Y$), making it ideal for validating whether TE correctly identifies both direction and lag.


2) Nonlinear System: Coupled Oscillators
To test estimator performance in a nonlinear regime, we simulate coupled oscillatory systems, where interactions are governed by nonlinear dynamics. These systems exhibit:
*	phase relationships,
*	nonlinear coupling,
*	richer dynamical structure compared to AR models.

Such signals are more representative of real-world processes (such as neural data), and therefore provide a more challenging benchmark for TE estimation methods.

⸻

2.3 Ground Truth and Experimental Control

A key advantage of simulated data is the availability of ground truth, which is used to:
	•	validate the detected direction of information flow,
	•	assess the accuracy in identifying the true interaction lag,
	•	quantify estimation errors under controlled perturbations.

Moreover, the simulation framework allows precise control over:
	•	Signal-to-Noise Ratio (SNR) (for robustness analysis),
	•	Sample size N (for bias and variance evaluation),
	•	Coupling strength (for sensitivity analysis).

⸻

2.4 Pre-analysis and Parameter Selection

Before computing Transfer Entropy, standard preprocessing steps are applied:
	•	Stationarity check using the Augmented Dickey-Fuller (ADF) test,
	•	Embedding delay (\tau) selection via the first minimum of Mutual Information,
	•	Embedding dimension (m) estimation using the False Nearest Neighbors (FNN) method.

These steps ensure that the reconstructed state space is appropriate for TE estimation and comparable across methods.


# ideas 

Assunzione,Test suggerito,Perché farlo?

Stazionarietà,Augmented Dickey-Fuller (ADF) Test,Per essere sicuro che il segnale non abbia trend o derive che invalidano la stima della densità di probabilità.

Linearità,Confronto TE vs Granger,"Calcola la TE con method='gauss' (lineare) e con method='kde' (non lineare). Se i risultati sono molto diversi, hai la prova che nel tuo segnale neurale c'è dinamica non lineare."

Ordine Markoviano,AIC/BIC o FNN,Per giustificare la scelta dell'embedding dimension m. Dimostri che non hai scelto m=1 a caso.

Significatività,Surrogati (Shuffling),Per distinguere la TE reale dal bias statistico dovuto alla lunghezza finita del segnale.


Se accade che...,La conclusione scientifica è...

TE (Gauss) ≈ TE (KDE),Il legame tra i neuroni è prevalentemente lineare. La semplicità vince.

TE (KDE) ≫ TE (Gauss),Esistono interazioni non lineari complesse (es. accoppiamento di fase) che un modello lineare non vede.

TE (Binning) è molto instabile,"Il segnale è troppo rumoroso o hai scelto pochi bin. Il binning soffre il ""bias di discretizzazione""."

TE (Copula) è la più robusta,"Le distribuzioni dei segnali non sono gaussiane (hanno code lunghe), ma la struttura di dipendenza è chiara."

A. Grafico di Robustezza al RumoreGenera un segnale AR con un lag noto. Aggiungi livelli crescenti di rumore bianco e calcola la TE con tutti i metodi.Cosa osservare: Quale metodo "crolla" per primo? Solitamente il binning è il più sensibile al rumore, mentre la Gaussian Copula è spesso la più resiliente.

B. Analisi del Tempo di CalcoloSe hai segnali molto lunghi, riporta il tempo di esecuzione.KDE: Molto preciso ma lento ($O(N^2)$).Binning/Gauss: Velocissimi.Conclusione: "Il KDE è preferibile per segmenti brevi e analisi di precisione, mentre il Binning è adatto per screening veloci su grandi dataset."

## 📋 To-Do List

#### **Fase 1: Generazione e Pre-analisi**
* [ ] **Creazione Ground Truth:** Genera i segnali AR (lineare) e Oscillatori Accoppiati (non-lineare).
* [ ] **Test di Stazionarietà (ADF):** Verifica che i segnali siano stabili.
* [ ] **Ottimizzazione Parametri (su dati puliti e lunghi):**
    * Trova l'**Embedding Delay ($\tau$)** con il *First Minimum of MI*.
    * Trova l'**Embedding Dimension ($m$)** con il *False Nearest Neighbors (FNN)*.

#### **Fase 2: Benchmark e Lag Scanning (L'Ideale)**
* [ ] **Lag Scanning:** Esegui una scansione del lag sorgente (1-20) su dati con alto SNR e molti campioni.
* [ ] **Confronto Metodi:** Calcola la TE con **KDE, Binning, Gaussian, Gaussian Copula** per identificare il picco di informazione.


#### **Fase 3: Stress Test - Robustezza al Rumore**
* [ ] **Noise Sweep:** Fissa $N$ (es. 5000) e varia il rumore (SNR da 30dB a 0dB).
* [ ] **Analisi Errore:** Per ogni metodo, osserva a quale livello di rumore il picco di TE scompare o diventa indistinguibile dal rumore di fondo.

#### **Fase 4: Stress Test - Lunghezza del Segnale ($N$)**
* [ ] **Sample Sweep:** Fissa il rumore (basso) e varia il numero di campioni $N$ (es. 500, 1000, 2000, 5000, 10000).
* [ ] **Bias Check:** Osserva come il valore della TE "gonfia" artificialmente quando $N$ è piccolo (bias statistico).


#### **Fase 5: Validazione Statistica**
* [ ] **Permutation Test:** Applica lo shuffling della sorgente per definire la soglia di confidenza (P-value) per i risultati ottenuti nelle fasi precedenti.

#### **Fase 6: Sintesi Comparativa**
* [ ] **Report Finale:** Crea grafici che mostrano l'accuratezza (errore rispetto al lag reale) in funzione del rumore e di $N$ per tutti e 4 i metodi.

---

### 📅 Tabella di Marcia

| Fase | Attività Principale | Focus Tecnico | Output Atteso |
| :--- | :--- | :--- | :--- |
| **Fase 1: Setup** | Generazione & Diagnostica | ADF, FNN, First Min MI | Parametri $m, \tau$ e stazionarietà confermata. |
| **Fase 2: Core** | Lag Scanning & Metodi | Confronto KDE, Binning, GC, Gauss | Grafico TE vs Lag (identificazione del picco). |
| **Fase 3: Rumore** | **Stress Test: Rumore** | Sweep SNR (0-30 dB) | Curve di decadimento della TE per ogni metodo. |
| **Fase 4: Dati** | **Stress Test: Campioni ($N$)** | Sweep $N$ (500-10.000) | Analisi del bias e stabilità della stima. |
| **Fase 5: Stat** | Permutation Test | Shuffling & Significatività | Barre di errore e soglie di confidenza (P < 0.05). |
| **Fase 6: Final** | Analisi Risultati | Confronto prestazioni totali | Tabella finale: "Qual è il metodo migliore?". |

---

### 💡 Un consiglio strategico per il tuo report
Quando confronterai i metodi sotto stress, probabilmente noterai che:
1.  **Gaussian Copula** sarà il più veloce e robusto al rumore.
2.  **KDE** sarà il più preciso per gli oscillatori ma "esploderà" in termini di tempo di calcolo se $N$ è troppo grande o fallirà se $N$ è troppo piccolo.
3.  **Binning** mostrerà un alto bias positivo quando $N$ è piccolo.

Presentare questi fallimenti è **più importante** che presentare solo i successi, perché dimostra che hai capito i limiti fisici dell'Information Theory.

Quale di questi due stress test ti preoccupa di più in termini di potenza di calcolo del tuo PC? (Se il KDE è lento, lo sweep su $N$ potrebbe richiedere ore).

In [3]:


# 1. Generazione
N = 5000
x_ar, y_ar = nl.generate_ar_coupled(N, lag=5)
x_osc, y_osc = nl.generate_oscillatory_coupled(N, lag=10)

# 2. Test Stazionarietà (ADF)
def check_stationarity(name, data):
    res = adfuller(data)
    print(f"[{name}] ADF Statistic: {res[0]:.3f}, p-value: {res[1]:.3e}")
    if res[1] < 0.05:
        print(f"--> {name} è STAZIONARIO")
    else:
        print(f"--> {name} NON è stazionario (Attenzione!)")

check_stationarity("AR_X", x_ar)
check_stationarity("Osc_Y", y_osc)

[AR_X] ADF Statistic: -29.637, p-value: 0.000e+00
--> AR_X è STAZIONARIO
[Osc_Y] ADF Statistic: -12.122, p-value: 1.814e-22
--> Osc_Y è STAZIONARIO


In [4]:
from neuro_lib.generators import generate_ar_coupled